In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0-pre — Environment verification
# FIX: inspect.getsource() fails on some PEP 660 editable installs
# (import-hook-based finder doesn't always support linecache lookup).
# Reading the .py files directly via open() avoids this entirely and
# is equally reliable for checking source text.
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, '-c', 'import depthcharge; print(depthcharge.__file__)'],
    capture_output=True, text=True
)
dc_path = result.stdout.strip()
print(f'depthcharge location: {dc_path}')

if 'site-packages' in dc_path:
    print('⚠ Editable install not active — installing now...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e',
         '/teamspace/studios/this_studio/depthcharge_changes',
         '--break-system-packages'],
        check=True
    )
    print('Done. Please RESTART the kernel and re-run this cell.')
else:
    print('✓ Editable install confirmed')

    dc_root = os.path.dirname(dc_path)  # .../depthcharge_changes/depthcharge

    analytes_path = os.path.join(dc_root, 'transformers', 'analytes.py')
    spectra_path  = os.path.join(dc_root, 'transformers', 'spectra.py')

    with open(analytes_path) as f:
        analytes_src = f.read()
    with open(spectra_path) as f:
        spectra_src = f.read()

    ok1 = 'flash_compatible' in analytes_src
    ok2 = 'new_zeros' in spectra_src

    print(f'  flash_compatible in analytes.py : {"✓" if ok1 else "✗ MISSING"}')
    print(f'  new_zeros in spectra.py         : {"✓" if ok2 else "✗ MISSING"}')

    if not (ok1 and ok2):
        raise RuntimeError(
            f'Branch changes not detected.\n'
            f'  analytes.py path: {analytes_path}\n'
            f'  spectra.py path : {spectra_path}'
        )
    print('\nReady → proceed to Cell 0 (NAR patch)')

depthcharge location: /teamspace/studios/this_studio/depthcharge_changes/depthcharge/__init__.py
✓ Editable install confirmed
  flash_compatible in analytes.py : ✓
  new_zeros in spectra.py         : ✓

Ready → proceed to Cell 0 (NAR patch)


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0 — NAR Patch (flash_compatible variant)
# FIX: verification block now reads function source via the `dis`/`__code__`
# co_filename + linecache fallback removed — instead we verify behaviorally
# (call the patched function's logic markers via a direct string check on
# its __code__.co_consts / closure, which doesn't depend on inspect at all)
# to avoid the same OSError seen in Cell 0-pre.
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, warnings
warnings.filterwarnings('ignore')


import torch
from casanovo.denovo.transformers import PeptideDecoder
from casanovo.denovo.model import Spec2Pep
from depthcharge.transformers import AnalyteTransformerDecoder

import depthcharge as _dc
print(f'depthcharge: {_dc.__file__}')
if 'site-packages' in _dc.__file__:
    raise RuntimeError(
        'Editable install not active.\n'
        'Run: pip install -e /teamspace/studios/this_studio/depthcharge_changes --break-system-packages'
    )
print('  ✓ local branch confirmed (not site-packages)\n')

# ── PATCH 1 — PeptideDecoder.embed ────────────────────────────────────
_ar_embed_original = AnalyteTransformerDecoder.embed

def _nar_embed(self, tokens, *args,
               memory,
               memory_key_padding_mask=None,
               memory_mask=None,
               tgt_mask=None,
               flash_compatible=False,
               _orig=_ar_embed_original,
               **kwargs):
    return _orig(
        self, tokens, *args,
        memory=memory,
        memory_key_padding_mask=memory_key_padding_mask,
        memory_mask=memory_mask,
        flash_compatible=True,
        **kwargs,
    )

assert _ar_embed_original is not _nar_embed, (
    'BUG: captured original is already the patch — restart kernel.')
PeptideDecoder.embed = _nar_embed

# ── PATCH 2 — Spec2Pep._forward_step (unchanged) ─────────────────────
def _nar_forward_step(self, batch):
    mzs, ints, precursors, seqs = self._process_batch(batch)
    dev = self.device
    mzs = mzs.to(dev); ints = ints.to(dev); precursors = precursors.to(dev)
    memories, mem_masks = self.encoder(mzs, ints)
    zero_tokens = (torch.zeros_like(seqs.to(dev)) if seqs is not None
                   else torch.zeros((mzs.shape[0], self.max_peptide_len),
                                    dtype=torch.long, device=dev))
    scores = self.decoder(tokens=zero_tokens, memory=memories,
                          memory_key_padding_mask=mem_masks,
                          precursors=precursors)
    return scores, seqs
Spec2Pep._forward_step = _nar_forward_step

# ── PATCH 3 — Spec2Pep.forward (unchanged) ───────────────────────────
def _nar_forward(self, batch): return self._forward_step(batch)
Spec2Pep.forward = _nar_forward

# ── Verification — behavioral, NOT inspect.getsource() ────────────────
# Checks the function's own __defaults__/__code__ structure directly,
# avoiding the linecache/OSError issue entirely.
import inspect as _inspect

_checks = {
    'PeptideDecoder.embed → _nar_embed'           : PeptideDecoder.embed is _nar_embed,
    'Spec2Pep._forward_step → _nar_forward_step'  : Spec2Pep._forward_step is _nar_forward_step,
    'Spec2Pep.forward → _nar_forward'             : Spec2Pep.forward is _nar_forward,
    'flash_compatible named param in _nar_embed'  : 'flash_compatible' in _inspect.signature(_nar_embed).parameters,
    'zero_tokens var in _nar_forward_step code'   : 'zero_tokens' in _nar_forward_step.__code__.co_varnames,
}
print('── NAR Patch Status ──────────────────────────────────────────')
for k, v in _checks.items():
    print(f'  {k:50s}: {"✓" if v else "✗ FAILED"}')
print('──────────────────────────────────────────────────────────────')
if not all(_checks.values()):
    raise RuntimeError('One or more patches failed.')
print('\nNAR patches applied ✓  (flash_compatible variant, duplicate-kwarg bug fixed)')

depthcharge: /teamspace/studios/this_studio/depthcharge_changes/depthcharge/__init__.py
  ✓ local branch confirmed (not site-packages)

── NAR Patch Status ──────────────────────────────────────────
  PeptideDecoder.embed → _nar_embed                 : ✓
  Spec2Pep._forward_step → _nar_forward_step        : ✓
  Spec2Pep.forward → _nar_forward                   : ✓
  flash_compatible named param in _nar_embed        : ✓
  zero_tokens var in _nar_forward_step code         : ✓
──────────────────────────────────────────────────────────────

NAR patches applied ✓  (flash_compatible variant, duplicate-kwarg bug fixed)


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Setup + Data + Model
# ═══════════════════════════════════════════════════════════════════════
import os, time, threading
import numpy as np, pandas as pd, datetime
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from torch.profiler import profile, ProfilerActivity, schedule, record_function
from torch.nn.attention import SDPBackend, sdpa_kernel
from contextlib import contextmanager
from tqdm import tqdm

WORK_DIR    = '/teamspace/studios/this_studio/nar_profiling'
RESULTS_DIR = '/teamspace/studios/this_studio/profiling_after_depthcharge_changes/results'
os.chdir(WORK_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Working directory : {os.getcwd()}')
print(f'Results directory : {RESULTS_DIR}')

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME   = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0
BF16_DTYPE     = torch.bfloat16
BF16_SUPPORTED = torch.cuda.is_bf16_supported() if DEVICE == 'cuda' else False

N_SUBSET         = 6000
N_TIMING_SPECTRA = 5000
BATCH_SIZES      = [1, 8, 32, 128, 512]
N_WARMUP_BATCHES = 10
N_PEAKS          = 150
PROF_WARMUP      = 20
PROF_ACTIVE      = 50

def _sync():
    if DEVICE == 'cuda': torch.cuda.synchronize()

@contextmanager
def _bf16_ctx():
    with torch.autocast(device_type='cuda', dtype=BF16_DTYPE):
        yield

def _mark_step():
    if DEVICE == 'cuda':
        torch.compiler.cudagraph_mark_step_begin()

def pad_or_select_to_fixed_peaks(mzs, ints, n=N_PEAKS):
    bs, L = mzs.shape
    if L == n: return mzs, ints, 0
    if L < n:
        return (torch.nn.functional.pad(mzs,  (0, n - L)),
                torch.nn.functional.pad(ints, (0, n - L)), 0)
    idx = ints.topk(n, dim=1).indices
    return mzs.gather(1, idx), ints.gather(1, idx), bs

def _start_gpu_monitor():
    samples, stop = [], threading.Event()
    def _fn():
        import subprocess as sp
        while not stop.is_set():
            r = sp.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used',
                        '--format=csv,noheader,nounits'],
                       capture_output=True, text=True)
            if r.returncode == 0:
                try:
                    u, m = r.stdout.strip().split(', ')
                    samples.append((int(u), float(m)/1024))
                except Exception: pass
            time.sleep(0.5)
    threading.Thread(target=_fn, daemon=True).start()
    return samples, stop

def _detect_attention_kernel(store, label):
    if not (DEVICE == 'cuda' and store.get('avgs')): return f'{label}: no CUDA data'
    flash = next((e for e in store['avgs'] if 'flash_attention' in e.key), None)
    eff   = next((e for e in store['avgs'] if 'efficient_attention' in e.key), None)
    if flash and eff:
        msg = f'{label}: BOTH flash ({flash.count}) + efficient ({eff.count})'
    elif flash:
        msg = f'{label}: FlashAttention ACTIVE ✓  ({flash.key}, {flash.count} calls)'
    elif eff:
        msg = f'{label}: memory-efficient only (NOT Flash)  ({eff.count} calls)'
    else:
        msg = f'{label}: no attention kernel found'
    print(msg); return msg

def _launch_count(store, n_active=PROF_ACTIVE):
    if not store.get('avgs'): return None
    e = next((e for e in store['avgs'] if e.key == 'cudaLaunchKernel'), None)
    return e.count // n_active if e else None

def _compiled_region_count(store):
    if not store.get('avgs'): return None
    return len([e for e in store['avgs'] if 'Torch-Compiled Region' in e.key])

def _save(obj, name):
    path = os.path.join(RESULTS_DIR, name)
    if hasattr(obj, 'savefig'):
        obj.savefig(path, dpi=150, bbox_inches='tight')
    else:
        obj.to_csv(path, index=False)
    print(f'Saved: {path}')
    return path

_tgt = lambda ms: 'MEETS ✓' if ms <= 10 else f'FAILS ({ms:.1f}ms)'

MGF_FILE   = 'multi-enzyme-simple.test.mgf'
SUBSET_MGF = 'subset_profile.mgf'
LANCE_DIR  = '.lance_cache'

if not os.path.exists(MGF_FILE):
    raise FileNotFoundError(f'{MGF_FILE} not found in {WORK_DIR}')
print(f'{MGF_FILE}  ({os.path.getsize(MGF_FILE)/1e6:.1f} MB)')

print('Spectra : 106,933')
print('Charge  : +1 to +8 | +2: 40,614  +3: 39,146')
print('m/z     : 301.2 – 1604.3')
print('Peaks   : 123 avg  (min 6, max 950)')

if not os.path.exists(SUBSET_MGF):
    raise FileNotFoundError(f'{SUBSET_MGF} not found in {WORK_DIR}')
print(f'Reusing: {SUBSET_MGF}  ({os.path.getsize(SUBSET_MGF)/1e6:.1f} MB)')

from casanovo.denovo import ModelRunner
from casanovo.denovo.model import Spec2Pep
from casanovo.denovo.dataloaders import DeNovoDataModule
from casanovo.config import Config
from casanovo.casanovo import _get_model_weights
import appdirs

config = Config(None)
cache_dir = Path(appdirs.user_cache_dir('casanovo', False, opinion=False))
model_path = _get_model_weights(cache_dir)
print(f'Model checkpoint: {model_path}')

runner = ModelRunner(config, model_path)
runner.initialize_tokenizer()
runner.initialize_model(train=False)
model = runner.model.eval().to(DEVICE)
MODEL_MAX_CHARGE = getattr(model, 'max_charge', config.max_charge)

print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params | '
      f'max_peptide_len={model.max_peptide_len} | max_charge={MODEL_MAX_CHARGE}')

import inspect as _inspect
from casanovo.denovo.transformers import PeptideDecoder
assert PeptideDecoder.embed is _nar_embed, 'NAR patch lost — re-run Cell 0!'
ok_fc = 'flash_compatible=True' in _inspect.getsource(_nar_embed)
print(f'NAR patch active ✓ | flash_compatible=True in patch: {"✓" if ok_fc else "✗"}')

_dm = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=1,
    tokenizer=runner.tokenizer,
    max_charge=MODEL_MAX_CHARGE,
    n_workers=0,
)
_dm.setup(stage='test', annotated=False)
_b              = next(iter(_dm.predict_dataloader()))
_mz, _it, _pr, _ = model._process_batch(_b)
print(f'First batch mzs={_mz.shape}  precs={_pr.shape} ✓')
print(f'Subset ready: {N_SUBSET} spectra  (timing target: {N_TIMING_SPECTRA})')

print(f'\nDevice : {DEVICE} | {GPU_NAME} | VRAM: {TOTAL_VRAM:.1f} GB')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda} | BF16: {BF16_SUPPORTED}')
print(f'Batch sizes: {BATCH_SIZES}')

Working directory : /teamspace/studios/this_studio/nar_profiling
Results directory : /teamspace/studios/this_studio/profiling_after_depthcharge_changes/results
multi-enzyme-simple.test.mgf  (300.9 MB)
Spectra : 106,933
Charge  : +1 to +8 | +2: 40,614  +3: 39,146
m/z     : 301.2 – 1604.3
Peaks   : 123 avg  (min 6, max 950)
Reusing: subset_profile.mgf  (16.9 MB)


Checkpoint directory not set in ModelRunner, no checkpoint files will be saved.
Configured residue(s) not in model alphabet: N[Deamidated], [Carbamyl]-, Q[Deamidated], [+25.980265]-, M[Oxidation], C[Carbamidomethyl], [Ammonia-loss]-, [Acetyl]-


Model checkpoint: /home/zeus/.cache/casanovo/casanovo_v5_0_0_v5_0_0.ckpt
Model: 47.9M params | max_peptide_len=100 | max_charge=4
NAR patch active ✓ | flash_compatible=True in patch: ✓


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

First batch mzs=torch.Size([1, 42])  precs=torch.Size([1, 3]) ✓
Subset ready: 6000 spectra  (timing target: 5000)

Device : cuda | NVIDIA L4 | VRAM: 23.6 GB
PyTorch: 2.7.1+cu128 | CUDA: 12.8 | BF16: True
Batch sizes: [1, 8, 32, 128, 512]


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — Baseline NAR FP32 Timing (eager, flash_compatible=True via patch)
# FP32 still uses memory-efficient attention (flash requires fp16/bf16),
# but this is now the clean baseline using the new patch mechanism.
# ═══════════════════════════════════════════════════════════════════════
_gpu_s, _gpu_stop = _start_gpu_monitor()
timing_fp32 = {}

for bs in BATCH_SIZES:
    print(f'\n══ FP32 Baseline  batch_size={bs:4d} ══')
    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.model.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)
    with torch.no_grad():
        for _w, _wb in enumerate(iter(_dm_bs.predict_dataloader())):
            if _w >= N_WARMUP_BATCHES: break
            _wm, _wi, _wp, _ = model._process_batch(_wb)
            _wm=_wm.to(DEVICE); _wi=_wi.to(DEVICE); _wp=_wp.to(DEVICE)
            _wme, _wmk = model.encoder(_wm, _wi)
            model.decoder(tokens=torch.zeros((_wm.shape[0], model.max_peptide_len),
                          dtype=torch.long, device=DEVICE),
                          memory=_wme, memory_key_padding_mask=_wmk, precursors=_wp)
    _sync()

    _t = {k: [] for k in ['fetch','h2d','enc','nar','write','total','tp']}
    _it = iter(_dm_bs.predict_dataloader()); n_spec = 0
    pbar = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')
    while n_spec < N_TIMING_SPECTRA:
        _sync(); t0 = time.perf_counter()
        try: batch = next(_it)
        except StopIteration: _it = iter(_dm_bs.predict_dataloader()); batch = next(_it)
        t_fetch = (time.perf_counter()-t0)*1000

        _sync(); t0 = time.perf_counter()
        mzs,ints,precs,_ = model._process_batch(batch)
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        _sync(); t_h2d=(time.perf_counter()-t0)*1000; ab=mzs.shape[0]

        with torch.no_grad():
            _sync(); t0=time.perf_counter()
            mem,mmk = model.encoder(mzs,ints)
            _sync(); t_enc=(time.perf_counter()-t0)*1000
            zt = torch.zeros((ab,model.max_peptide_len),dtype=torch.long,device=DEVICE)
            _sync(); t0=time.perf_counter()
            scores = model.decoder(tokens=zt,memory=mem,
                                   memory_key_padding_mask=mmk,precursors=precs)
            _sync(); t_nar=(time.perf_counter()-t0)*1000

        t0=time.perf_counter()
        pred=scores.argmax(dim=-1).cpu(); _=[{'tokens':t.tolist()} for t in pred]
        t_write=(time.perf_counter()-t0)*1000
        tt=t_fetch+t_h2d+t_enc+t_nar+t_write
        for k,v in zip(['fetch','h2d','enc','nar','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_enc/ab,t_nar/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec+=ab; pbar.update(ab)
        if n_spec>=N_TIMING_SPECTRA: break
    pbar.close()

    p = lambda a,q: float(np.percentile(a,q))
    timing_fp32[bs] = {'n_spec':n_spec,'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),
        'enc':np.mean(_t['enc']),'nar':np.mean(_t['nar']),'write':np.mean(_t['write']),
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s=timing_fp32[bs]
    print(f'  total={s["total"]:.2f}ms  enc={s["enc"]:.2f}ms  nar={s["nar"]:.2f}ms  tp={s["tp"]:.1f}spec/s')
    if DEVICE=='cuda': torch.cuda.empty_cache()

_gpu_stop.set(); time.sleep(1.0)
gpu_util_fp32=np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_fp32=np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

b1=timing_fp32[1]; rb=b1['raw']
df_stage_fp32 = pd.DataFrame([
    {'Stage':'DataLoader fetch','mean_ms':b1['fetch'],'p50_ms':np.percentile(rb['fetch'],50),'p95_ms':np.percentile(rb['fetch'],95)},
    {'Stage':'H2D transfer',    'mean_ms':b1['h2d'],  'p50_ms':np.percentile(rb['h2d'],50),  'p95_ms':np.percentile(rb['h2d'],95)},
    {'Stage':'SpectrumEncoder', 'mean_ms':b1['enc'],  'p50_ms':np.percentile(rb['enc'],50),  'p95_ms':np.percentile(rb['enc'],95)},
    {'Stage':'NAR Decoder',     'mean_ms':b1['nar'],  'p50_ms':np.percentile(rb['nar'],50),  'p95_ms':np.percentile(rb['nar'],95)},
    {'Stage':'Output write',    'mean_ms':b1['write'],'p50_ms':np.percentile(rb['write'],50),'p95_ms':np.percentile(rb['write'],95)},
    {'Stage':'TOTAL',           'mean_ms':b1['total'],'p50_ms':b1['p50'],                   'p95_ms':b1['p95']},
]).round(3)
df_tp_fp32 = pd.DataFrame([{'batch_size':bs,'total_ms':timing_fp32[bs]['total'],
    'tp_spec_s':timing_fp32[bs]['tp'],'enc_ms':timing_fp32[bs]['enc'],
    'nar_ms':timing_fp32[bs]['nar'],'p50':timing_fp32[bs]['p50'],
    'p95':timing_fp32[bs]['p95']} for bs in BATCH_SIZES]).round(3)
print(f'\n── Stage breakdown (FP32, bs=1, {b1["n_spec"]} spectra) ──')
print(df_stage_fp32.to_string(index=False))
print(f'\n── Throughput (FP32) ──\n{df_tp_fp32.to_string(index=False)}')
print(f'\nGPU util: {gpu_util_fp32:.0f}%  |  Peak VRAM: {gpu_vram_fp32:.2f} GB')
_tgt = lambda ms: 'MEETS ✓' if ms<=10 else f'FAILS ({ms:.1f}ms)'
print(f'10ms target (bs=1): {_tgt(b1["total"])}')
df_stage_fp32.to_csv('results/fp32_stage_bs1.csv',index=False)
df_tp_fp32.to_csv('results/fp32_throughput.csv',index=False)


══ FP32 Baseline  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [01:44<00:00, 47.84spec/s]


  total=20.53ms  enc=7.91ms  nar=10.96ms  tp=49.4spec/s

══ FP32 Baseline  batch_size=   8 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

...

  bs=128: 5120spec [00:10, 507.36spec/s]                        

  total=1.96ms  enc=0.56ms  nar=1.20ms  tp=510.9spec/s

══ FP32 Baseline  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=512: 5120spec [00:10, 469.63spec/s]                        


  total=2.13ms  enc=0.67ms  nar=1.25ms  tp=470.5spec/s

── Stage breakdown (FP32, bs=1, 5000 spectra) ──
           Stage  mean_ms  p50_ms  p95_ms
DataLoader fetch    1.352   1.280   1.905
    H2D transfer    0.213   0.201   0.304
 SpectrumEncoder    7.906   7.440  11.644
     NAR Decoder   10.957  10.425  14.890
    Output write    0.104   0.095   0.135
           TOTAL   20.532  19.528  28.500

── Throughput (FP32) ──
 batch_size  total_ms  tp_spec_s  enc_ms  nar_ms    p50    p95
          1    20.532     49.445   7.906  10.957 19.528 28.500
          8     2.959    344.035   1.055   1.495  2.778  4.081
         32     1.811    553.260   0.596   0.950  1.797  1.955
        128     1.959    510.868   0.555   1.199  1.953  2.054
        512     2.126    470.501   0.674   1.247  2.130  2.182

GPU util: 34%  |  Peak VRAM: 2.51 GB
10ms target (bs=1): FAILS (20.5ms)


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — BF16 + FlashAttention Timing
# FIX: the previous version reused _mz/_it/_pr from Cell 1's smoke-test,
# but Cell 2's timing loop reassigns the name `_it` to a DataLoader
# iterator object (`_it = iter(...)`), silently shadowing the original
# intensity tensor. Fetching a fresh batch here avoids depending on any
# global state from earlier cells.
# ═══════════════════════════════════════════════════════════════════════
assert BF16_SUPPORTED, 'BF16 not supported on this GPU.'

# ── Quick flash probe (fetch a fresh real spectrum, don't reuse globals) ─
_probe_dm = DeNovoDataModule(
    lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
    eval_batch_size=1, tokenizer=runner.tokenizer,
    max_charge=MODEL_MAX_CHARGE, n_workers=0,
)
_probe_dm.setup(stage='test', annotated=False)
_probe_batch = next(iter(_probe_dm.predict_dataloader()))
_probe_mz, _probe_it, _probe_pr, _ = model._process_batch(_probe_batch)

_pb_mz, _pb_it, _pb_pr = _probe_mz.to(DEVICE), _probe_it.to(DEVICE), _probe_pr.to(DEVICE)
_pz = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)
FLASH_ACCEPTED = False
try:
    with torch.no_grad(), _bf16_ctx():
        with sdpa_kernel(backends=[SDPBackend.FLASH_ATTENTION]):
            _pme, _pmk = model.encoder(_pb_mz, _pb_it)
            model.decoder(tokens=_pz, memory=_pme,
                          memory_key_padding_mask=_pmk, precursors=_pb_pr)
    FLASH_ACCEPTED = True
except RuntimeError as _e:
    _flash_err = str(_e)[:120]
_sync()

print('── FlashAttention probe (BF16 + flash_compatible=True) ──────')
if FLASH_ACCEPTED:
    print('  RESULT: Flash ACCEPTED ✓ — FlashAttention now active for decoder self-attention.')
else:
    print(f'  RESULT: Flash rejected — {_flash_err}')
    print('  NOTE: Decoder cross-attention (memory_key_padding_mask present) '
          'will still use memory-efficient attention.')
print('─────────────────────────────────────────────────────────────\n')

# ── Timing ────────────────────────────────────────────────────────────
_gpu_s, _gpu_stop = _start_gpu_monitor()
timing_bf16 = {}

for bs in BATCH_SIZES:
    print(f'\n══ BF16+Flash  batch_size={bs:4d} ══')
    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)
    with torch.no_grad(), _bf16_ctx():
        for _w,_wb in enumerate(iter(_dm_bs.predict_dataloader())):
            if _w>=N_WARMUP_BATCHES: break
            _wm,_wi,_wp,_ = model._process_batch(_wb)
            _wm=_wm.to(DEVICE); _wi=_wi.to(DEVICE); _wp=_wp.to(DEVICE)
            _wme,_wmk = model.encoder(_wm,_wi)
            model.decoder(tokens=torch.zeros((_wm.shape[0],model.max_peptide_len),
                          dtype=torch.long,device=DEVICE),
                          memory=_wme,memory_key_padding_mask=_wmk,precursors=_wp)
    _sync()

    _t={k:[] for k in ['fetch','h2d','enc','nar','write','total','tp']}
    _loader_iter=iter(_dm_bs.predict_dataloader()); n_spec=0
    pbar=tqdm(total=N_TIMING_SPECTRA,desc=f'  bs={bs}',unit='spec')
    while n_spec<N_TIMING_SPECTRA:
        _sync(); t0=time.perf_counter()
        try: batch=next(_loader_iter)
        except StopIteration: _loader_iter=iter(_dm_bs.predict_dataloader()); batch=next(_loader_iter)
        t_fetch=(time.perf_counter()-t0)*1000
        _sync(); t0=time.perf_counter()
        mzs,ints,precs,_=model._process_batch(batch)
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        _sync(); t_h2d=(time.perf_counter()-t0)*1000; ab=mzs.shape[0]
        with torch.no_grad(), _bf16_ctx():
            _sync(); t0=time.perf_counter()
            mem,mmk=model.encoder(mzs,ints)
            _sync(); t_enc=(time.perf_counter()-t0)*1000
            zt=torch.zeros((ab,model.max_peptide_len),dtype=torch.long,device=DEVICE)
            _sync(); t0=time.perf_counter()
            scores=model.decoder(tokens=zt,memory=mem,
                                 memory_key_padding_mask=mmk,precursors=precs)
            _sync(); t_nar=(time.perf_counter()-t0)*1000
        t0=time.perf_counter()
        pred=scores.argmax(dim=-1).cpu(); _=[{'tokens':t.tolist()} for t in pred]
        t_write=(time.perf_counter()-t0)*1000
        tt=t_fetch+t_h2d+t_enc+t_nar+t_write
        for k,v in zip(['fetch','h2d','enc','nar','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_enc/ab,t_nar/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec+=ab; pbar.update(ab)
        if n_spec>=N_TIMING_SPECTRA: break
    pbar.close()
    p=lambda a,q: float(np.percentile(a,q))
    timing_bf16[bs]={'n_spec':n_spec,'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),
        'enc':np.mean(_t['enc']),'nar':np.mean(_t['nar']),'write':np.mean(_t['write']),
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s=timing_bf16[bs]
    print(f'  total={s["total"]:.2f}ms  enc={s["enc"]:.2f}ms  nar={s["nar"]:.2f}ms  tp={s["tp"]:.1f}spec/s')
    spd=timing_fp32[bs]['total']/max(s['total'],0.001)
    print(f'  vs FP32: {spd:.2f}×')
    if DEVICE=='cuda': torch.cuda.empty_cache()

_gpu_stop.set(); time.sleep(1.0)
gpu_util_bf16=np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_bf16=np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

c1=timing_bf16[1]; rc=c1['raw']
df_stage_bf16=pd.DataFrame([
    {'Stage':'DataLoader fetch',   'mean_ms':c1['fetch'],'p50_ms':np.percentile(rc['fetch'],50),'p95_ms':np.percentile(rc['fetch'],95)},
    {'Stage':'H2D transfer',       'mean_ms':c1['h2d'],  'p50_ms':np.percentile(rc['h2d'],50),  'p95_ms':np.percentile(rc['h2d'],95)},
    {'Stage':'SpectrumEncoder(BF16)','mean_ms':c1['enc'],'p50_ms':np.percentile(rc['enc'],50),  'p95_ms':np.percentile(rc['enc'],95)},
    {'Stage':'NAR Decoder(BF16+Flash)','mean_ms':c1['nar'],'p50_ms':np.percentile(rc['nar'],50),'p95_ms':np.percentile(rc['nar'],95)},
    {'Stage':'Output write',       'mean_ms':c1['write'],'p50_ms':np.percentile(rc['write'],50),'p95_ms':np.percentile(rc['write'],95)},
    {'Stage':'TOTAL',              'mean_ms':c1['total'],'p50_ms':c1['p50'],                   'p95_ms':c1['p95']},
]).round(3)
df_tp_bf16=pd.DataFrame([{'batch_size':bs,'total_ms':timing_bf16[bs]['total'],
    'tp_spec_s':timing_bf16[bs]['tp'],'enc_ms':timing_bf16[bs]['enc'],
    'nar_ms':timing_bf16[bs]['nar'],'p50':timing_bf16[bs]['p50'],
    'p95':timing_bf16[bs]['p95'],'vs_fp32':f"{timing_fp32[bs]['total']/max(timing_bf16[bs]['total'],0.001):.2f}x"}
    for bs in BATCH_SIZES]).round(3)
print(f'\n── Stage breakdown (BF16+Flash, bs=1) ──\n{df_stage_bf16.to_string(index=False)}')
print(f'\n── Throughput (BF16+Flash) ──\n{df_tp_bf16.to_string(index=False)}')
print(f'\nGPU util: {gpu_util_bf16:.0f}%  |  Peak VRAM: {gpu_vram_bf16:.2f} GB')
print(f'10ms target (bs=1): {_tgt(c1["total"])}')
_save(df_stage_bf16, 'bf16_stage_bs1.csv')
_save(df_tp_bf16, 'bf16_throughput.csv')

subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

── FlashAttention probe (BF16 + flash_compatible=True) ──────
  RESULT: Flash rejected — No available kernel. Aborting execution.
  NOTE: Decoder cross-attention (memory_key_padding_mask present) will still use memory-efficient attention.
─────────────────────────────────────────────────────────────


══ BF16+Flash  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [02:12<00:00, 37.69spec/s]


  total=25.88ms  enc=9.52ms  nar=14.59ms  tp=39.3spec/s
  vs FP32: 0.79×

══ BF16+Flash  batch_size=   8 ══


...

  total=1.05ms  enc=0.36ms  nar=0.47ms  tp=950.3spec/s
  vs FP32: 1.86×

══ BF16+Flash  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=512: 5120spec [00:05, 967.51spec/s]                        


  total=1.03ms  enc=0.40ms  nar=0.45ms  tp=971.3spec/s
  vs FP32: 2.06×

── Stage breakdown (BF16+Flash, bs=1) ──
                  Stage  mean_ms  p50_ms  p95_ms
       DataLoader fetch    1.412   1.338   1.983
           H2D transfer    0.225   0.211   0.323
  SpectrumEncoder(BF16)    9.525   8.887  14.184
NAR Decoder(BF16+Flash)   14.592  13.810  19.694
           Output write    0.122   0.114   0.174
                  TOTAL   25.876  24.497  34.919

── Throughput (BF16+Flash) ──
 batch_size  total_ms  tp_spec_s  enc_ms  nar_ms    p50    p95 vs_fp32
          1    25.876     39.256   9.525  14.592 24.497 34.919   0.79x
          8     3.661    277.843   1.259   1.988  3.436  4.997   0.81x
         32     1.133    900.376   0.350   0.527  1.056  1.467   1.60x
        128     1.054    950.256   0.363   0.471  1.034  1.135   1.86x
        512     1.030    971.274   0.401   0.452  1.029  1.044   2.06x

GPU util: 20%  |  Peak VRAM: 2.76 GB
10ms target (bs=1): FAILS (25.9ms)
Saved: /teams

'/teamspace/studios/this_studio/profiling_after_depthcharge_changes/results/bf16_throughput.csv'

In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — torch.compile + BF16 + FlashAttention Timing
# Two depthcharge fixes now enable better CUDA Graph capture:
#   (1) spectra.py: new_zeros() instead of torch.tensor([[False]]*n)
#       → eliminates the data-dependent graph break in SpectrumEncoder
#   (2) flash_compatible=True (via patch): no mask tensors in decoder
#       → cleaner decoder graph with no data-dependent tensor injection
# Still requires N_PEAKS=150 fixed shapes for CUDA Graph replay.
# Still requires cudagraph_mark_step_begin() when chaining two
# separately compiled modules (encoder → decoder).
# ═══════════════════════════════════════════════════════════════════════
torch._dynamo.config.cache_size_limit = 32

compiled_encoder = torch.compile(model.encoder, mode='reduce-overhead')
compiled_decoder = torch.compile(model.decoder, mode='reduce-overhead')
print('compiled_encoder / compiled_decoder created (mode=reduce-overhead)')
print('First call at each batch size triggers JIT + CUDA Graph capture.')

_gpu_s, _gpu_stop = _start_gpu_monitor()
timing_compiled = {}

for bs in BATCH_SIZES:
    print(f'\n══ Compiled+BF16+Flash  batch_size={bs:4d} ══')
    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.model.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)

    print(f'  Compiling + capturing CUDA Graph for bs={bs}…')
    _t0_compile = time.perf_counter()
    with torch.no_grad(), _bf16_ctx():
        for _w,_wb in enumerate(iter(_dm_bs.predict_dataloader())):
            if _w>=N_WARMUP_BATCHES: break
            _wm,_wi,_wp,_ = model._process_batch(_wb)
            _wm=_wm.to(DEVICE); _wi=_wi.to(DEVICE); _wp=_wp.to(DEVICE)
            _wm,_wi,_ = pad_or_select_to_fixed_peaks(_wm,_wi)
            _mark_step()
            _wme,_wmk = compiled_encoder(_wm,_wi)
            compiled_decoder(tokens=torch.zeros((_wm.shape[0],model.max_peptide_len),
                             dtype=torch.long,device=DEVICE),
                             memory=_wme,memory_key_padding_mask=_wmk,precursors=_wp)
    _sync()
    compile_s = time.perf_counter()-_t0_compile
    print(f'  Compile+capture done in {compile_s:.1f}s (excluded from timings below)')

    _t={k:[] for k in ['fetch','h2d','enc','nar','write','total','tp']}
    _it=iter(_dm_bs.predict_dataloader()); n_spec=0
    pbar=tqdm(total=N_TIMING_SPECTRA,desc=f'  bs={bs}',unit='spec')
    while n_spec<N_TIMING_SPECTRA:
        _sync(); t0=time.perf_counter()
        try: batch=next(_it)
        except StopIteration: _it=iter(_dm_bs.predict_dataloader()); batch=next(_it)
        t_fetch=(time.perf_counter()-t0)*1000
        _sync(); t0=time.perf_counter()
        mzs,ints,precs,_=model._process_batch(batch)
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        mzs,ints,_ntrunc=pad_or_select_to_fixed_peaks(mzs,ints)
        _sync(); t_h2d=(time.perf_counter()-t0)*1000; ab=mzs.shape[0]
        with torch.no_grad(), _bf16_ctx():
            _mark_step()
            _sync(); t0=time.perf_counter()
            mem,mmk=compiled_encoder(mzs,ints)
            _sync(); t_enc=(time.perf_counter()-t0)*1000
            zt=torch.zeros((ab,model.max_peptide_len),dtype=torch.long,device=DEVICE)
            _sync(); t0=time.perf_counter()
            scores=compiled_decoder(tokens=zt,memory=mem,
                                    memory_key_padding_mask=mmk,precursors=precs)
            _sync(); t_nar=(time.perf_counter()-t0)*1000
        t0=time.perf_counter()
        pred=scores.argmax(dim=-1).cpu(); _=[{'tokens':t.tolist()} for t in pred]
        t_write=(time.perf_counter()-t0)*1000
        tt=t_fetch+t_h2d+t_enc+t_nar+t_write
        for k,v in zip(['fetch','h2d','enc','nar','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_enc/ab,t_nar/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec+=ab; pbar.update(ab)
        if n_spec>=N_TIMING_SPECTRA: break
    pbar.close()
    p=lambda a,q:float(np.percentile(a,q))
    timing_compiled[bs]={'n_spec':n_spec,'compile_s':compile_s,
        'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),
        'enc':np.mean(_t['enc']),'nar':np.mean(_t['nar']),'write':np.mean(_t['write']),
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s=timing_compiled[bs]
    spd=timing_fp32[bs]['total']/max(s['total'],0.001)
    print(f'  total={s["total"]:.2f}ms  enc={s["enc"]:.2f}ms  nar={s["nar"]:.2f}ms  '
          f'tp={s["tp"]:.1f}spec/s  vs FP32: {spd:.2f}×')
    if DEVICE=='cuda': torch.cuda.empty_cache()

_gpu_stop.set(); time.sleep(1.0)
gpu_util_comp=np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_comp=np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

e1=timing_compiled[1]; re=e1['raw']
df_stage_comp=pd.DataFrame([
    {'Stage':'DataLoader fetch',       'mean_ms':e1['fetch'],'p50_ms':np.percentile(re['fetch'],50),'p95_ms':np.percentile(re['fetch'],95)},
    {'Stage':'H2D+fixed-peak pad',     'mean_ms':e1['h2d'],  'p50_ms':np.percentile(re['h2d'],50),  'p95_ms':np.percentile(re['h2d'],95)},
    {'Stage':'Encoder(Compiled+BF16)', 'mean_ms':e1['enc'],  'p50_ms':np.percentile(re['enc'],50),  'p95_ms':np.percentile(re['enc'],95)},
    {'Stage':'Decoder(Compiled+Flash)','mean_ms':e1['nar'],  'p50_ms':np.percentile(re['nar'],50),  'p95_ms':np.percentile(re['nar'],95)},
    {'Stage':'Output write',           'mean_ms':e1['write'],'p50_ms':np.percentile(re['write'],50),'p95_ms':np.percentile(re['write'],95)},
    {'Stage':'TOTAL',                  'mean_ms':e1['total'],'p50_ms':e1['p50'],                   'p95_ms':e1['p95']},
]).round(3)
df_tp_comp=pd.DataFrame([{'batch_size':bs,'total_ms':timing_compiled[bs]['total'],
    'tp_spec_s':timing_compiled[bs]['tp'],'enc_ms':timing_compiled[bs]['enc'],
    'nar_ms':timing_compiled[bs]['nar'],'compile_s':round(timing_compiled[bs]['compile_s'],1),
    'vs_fp32':f"{timing_fp32[bs]['total']/max(timing_compiled[bs]['total'],0.001):.2f}x"}
    for bs in BATCH_SIZES]).round(3)
print(f'\n── Stage breakdown (Compiled+BF16+Flash, bs=1) ──\n{df_stage_comp.to_string(index=False)}')
print(f'\n── Throughput (Compiled+BF16+Flash) ──\n{df_tp_comp.to_string(index=False)}')
print(f'\nGPU util: {gpu_util_comp:.0f}%  |  Peak VRAM: {gpu_vram_comp:.2f} GB')
print(f'10ms target (bs=1): {_tgt(e1["total"])}')
df_stage_comp.to_csv('results/compiled_stage_bs1.csv',index=False)
df_tp_comp.to_csv('results/compiled_throughput.csv',index=False)

compiled_encoder / compiled_decoder created (mode=reduce-overhead)
First call at each batch size triggers JIT + CUDA Graph capture.

══ Compiled+BF16+Flash  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compiling + capturing CUDA Graph for bs=1…


W0630 10:48:40.383000 288375 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/_inductor/utils.py:1250] [0/0_1] Not enough SMs to use max_autotune_gemm mode


  Compile+capture done in 23.3s (excluded from timings below)


...

  total=0.98ms  enc=0.35ms  nar=0.42ms  tp=1018.8spec/s  vs FP32: 1.99×

══ Compiled+BF16+Flash  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compiling + capturing CUDA Graph for bs=512…
  Compile+capture done in 5.6s (excluded from timings below)


  bs=512: 5120spec [00:04, 1075.07spec/s]                        


  total=0.93ms  enc=0.38ms  nar=0.35ms  tp=1081.4spec/s  vs FP32: 2.30×

── Stage breakdown (Compiled+BF16+Flash, bs=1) ──
                  Stage  mean_ms  p50_ms  p95_ms
       DataLoader fetch    1.281   1.201   1.800
     H2D+fixed-peak pad    0.298   0.292   0.453
 Encoder(Compiled+BF16)    9.455   8.752  14.199
Decoder(Compiled+Flash)    1.765   1.722   2.048
           Output write    0.124   0.115   0.169
                  TOTAL   12.923  12.110  18.602

── Throughput (Compiled+BF16+Flash) ──
 batch_size  total_ms  tp_spec_s  enc_ms  nar_ms  compile_s vs_fp32
          1    12.923     78.853   9.455   1.765       23.3   1.59x
          8     2.464    411.519   1.290   0.750       33.4   1.20x
         32     0.964   1048.651   0.359   0.339        1.6   1.88x
        128     0.983   1018.750   0.348   0.419        2.4   1.99x
        512     0.925   1081.392   0.381   0.347        5.6   2.30x

GPU util: 21%  |  Peak VRAM: 3.99 GB
10ms target (bs=1): FAILS (12.9ms)


In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — torch.profiler: FP32 vs BF16+Flash vs Compiled+BF16+Flash
# Key diagnostics:
#   (1) Attention kernel: flash vs efficient (per variant)
#   (2) Torch-Compiled Regions: how many? (should be fewer now — spectra
#       encoder graph-break fix + no mask injection in decoder)
#   (3) Kernel launches per spectrum
# ═══════════════════════════════════════════════════════════════════════
ACTS = ([ProfilerActivity.CPU, ProfilerActivity.CUDA]
        if DEVICE == 'cuda' else [ProfilerActivity.CPU])
N_PROF = PROF_WARMUP + PROF_ACTIVE

print(f'Pre-fetching {N_PROF} bs=1 batches…')
_dm_prof = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                            eval_batch_size=1, tokenizer=runner.model.tokenizer,
                            max_charge=MODEL_MAX_CHARGE, n_workers=0)
_dm_prof.setup(stage='test', annotated=False)

_prof_batches, _prof_batches_fixed = [], []
for _b in _dm_prof.predict_dataloader():
    _mz2, _it2, _pr2, _ = model._process_batch(_b)
    _prof_batches.append((_mz2.to(DEVICE), _it2.to(DEVICE), _pr2.to(DEVICE)))
    _mzf, _itf, _ = pad_or_select_to_fixed_peaks(_mz2.to(DEVICE), _it2.to(DEVICE))
    _prof_batches_fixed.append((_mzf, _itf, _pr2.to(DEVICE)))
    if len(_prof_batches) >= N_PROF: break
while len(_prof_batches) < N_PROF:
    _prof_batches.extend(_prof_batches[:N_PROF-len(_prof_batches)])
    _prof_batches_fixed.extend(_prof_batches_fixed[:N_PROF-len(_prof_batches_fixed)])
print(f'Using {len(_prof_batches)} batches')

_zt_prof = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)

def _run_prof(label, trace_path, txt_path, batches, ctx_fn, enc_fn, dec_fn,
              use_mark_step=False, n_warm=10):
    with torch.no_grad(), ctx_fn():
        for _mz2,_it2,_pr2 in batches[:n_warm]:
            if use_mark_step: _mark_step()
            _me,_mk = enc_fn(_mz2,_it2)
            dec_fn(tokens=_zt_prof,memory=_me,memory_key_padding_mask=_mk,precursors=_pr2)
    _sync()
    _store={}
    def _on_ready(p):
        p.export_chrome_trace(trace_path)
        _store['tbl']  = p.key_averages().table(sort_by='cpu_time_total', row_limit=12)
        _store['avgs'] = p.key_averages()
    with profile(activities=ACTS, record_shapes=True,
                 schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
                 on_trace_ready=_on_ready) as p:
        with torch.no_grad(), ctx_fn():
            for _mz2,_it2,_pr2 in batches:
                if use_mark_step: _mark_step()
                with record_function(label):
                    _me,_mk = enc_fn(_mz2,_it2)
                    dec_fn(tokens=_zt_prof,memory=_me,
                           memory_key_padding_mask=_mk,precursors=_pr2)
                _sync(); p.step()
    print(_store.get('tbl','(no data)'))
    with open(txt_path,'w') as f:
        f.write(f'{label}  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
        f.write('='*64+'\n'+str(_store.get('tbl','no data')))
    print(f'Chrome trace → {trace_path}')
    if DEVICE=='cuda': torch.cuda.synchronize(); torch.cuda.empty_cache()
    return _store

@contextmanager
def _fp32_ctx():
    yield   # plain eager FP32 — no autocast

# ── A) Baseline FP32 ──────────────────────────────────────────────────
print('\n── A) torch.profiler: Baseline FP32 (eager) ─────────────────')
_store_a = _run_prof('fp32_full_forward', 'results/trace_fp32.json',
                     'results/profiler_fp32.txt', _prof_batches, _fp32_ctx,
                     model.encoder, model.decoder)
_attn_a = _detect_attention_kernel(_store_a, 'A) FP32')
_nkern_a = _launch_count(_store_a)
_nreg_a  = _compiled_region_count(_store_a)

# ── B) BF16 + Flash ───────────────────────────────────────────────────
print('\n── B) torch.profiler: BF16 + FlashAttention (eager) ─────────')
_store_b = _run_prof('bf16_flash_forward', 'results/trace_bf16.json',
                     'results/profiler_bf16.txt', _prof_batches, _bf16_ctx,
                     model.encoder, model.decoder)
_attn_b = _detect_attention_kernel(_store_b, 'B) BF16+Flash')
_nkern_b = _launch_count(_store_b)
_nreg_b  = _compiled_region_count(_store_b)

# ── C) Compiled + BF16 + Flash ────────────────────────────────────────
print('\n── C) torch.profiler: Compiled + BF16 + Flash ───────────────')
_store_c = _run_prof('compiled_bf16_flash', 'results/trace_compiled.json',
                     'results/profiler_compiled.txt', _prof_batches_fixed, _bf16_ctx,
                     compiled_encoder, compiled_decoder, use_mark_step=True)
_attn_c = _detect_attention_kernel(_store_c, 'C) Compiled+BF16+Flash')
_nkern_c = _launch_count(_store_c)
_nreg_c  = _compiled_region_count(_store_c)

print('\n── Profiler Comparison Summary ───────────────────────────────')
print(f'{"Metric":<35} {"A:FP32":>12} {"B:BF16+Flash":>14} {"C:Compiled":>12}')
print('-'*76)
print(f'{"Kernel launches/spectrum":<35} {str(_nkern_a):>12} {str(_nkern_b):>14} {str(_nkern_c):>12}')
print(f'{"Torch-Compiled Regions":<35} {str(_nreg_a or "N/A"):>12} {str(_nreg_b or "N/A"):>14} {str(_nreg_c or "N/A"):>12}')
print(f'{"Attention kernel":<35} {"efficient":>12} '
      f'{"flash✓" if FLASH_ACCEPTED else "efficient":>14} '
      f'{"flash✓" if FLASH_ACCEPTED else "efficient":>12}')

Pre-fetching 70 bs=1 batches…


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

Using 70 batches

── A) torch.profiler: Baseline FP32 (eager) ─────────────────
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.32%       6.860ms       100.00%        2.125s      42.501ms       0.000us         0.00%     146.715ms       2.934ms            50  
                                      fp32_full_forward        22.59%     480.066ms        99.60%        2.117s      42.331ms  

In [9]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — Comparison Plots: FP32 vs BF16+Flash vs Compiled+BF16+Flash
# ═══════════════════════════════════════════════════════════════════════
COLORS = {'fp32': '#D85A30', 'bf16': '#1D9E75', 'compiled': '#3A7FC1'}
LABELS = {
    'fp32'    : 'Baseline FP32 (eager)',
    'bf16'    : 'BF16 + Flash (eager)',
    'compiled': 'Compiled + BF16 + Flash',
}
_xi   = list(range(len(BATCH_SIZES)))
_xlbl = [str(b) for b in BATCH_SIZES]
_tps  = {k: [timing[bs]['tp']    for bs in BATCH_SIZES]
         for k, timing in [('fp32',timing_fp32),('bf16',timing_bf16),('compiled',timing_compiled)]}
_lats = {k: [timing[bs]['total'] for bs in BATCH_SIZES]
         for k, timing in [('fp32',timing_fp32),('bf16',timing_bf16),('compiled',timing_compiled)]}

# ── Figure 1 — Stage breakdown bs=1 ──────────────────────────────────
fig1, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig1.suptitle('Stage Breakdown (bs=1) — Three Variants', fontweight='bold')
for ax, (key, df, total) in zip(axes, [
    ('fp32',     df_stage_fp32, timing_fp32[1]['total']),
    ('bf16',     df_stage_bf16, timing_bf16[1]['total']),
    ('compiled', df_stage_comp, timing_compiled[1]['total']),
]):
    _stages = ['Fetch','H2D','Encoder','Decoder','Write']
    _vals   = df[df['Stage'] != 'TOTAL']['mean_ms'].values
    bars = ax.bar(_stages, _vals, color=COLORS[key], edgecolor='none', width=0.55)
    for b, v in zip(bars, _vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.15, f'{v:.2f}',
                ha='center', fontsize=8)
    ax.set_title(f'{LABELS[key]}\nTotal: {total:.1f} ms/spec')
    ax.set_ylabel('ms / spectrum')
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/stage_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: results/stage_comparison.png')

# ── Figure 2 — Throughput + Latency vs batch size ────────────────────
fig2, (ax_tp, ax_lat) = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle('NAR Performance vs Batch Size — Three Variants', fontweight='bold')
for key in ['fp32','bf16','compiled']:
    ax_tp.plot(_xi, _tps[key], 'o-', color=COLORS[key], lw=2, ms=8, label=LABELS[key])
    ax_lat.plot(_xi, _lats[key], 'o-', color=COLORS[key], lw=2, ms=8, label=LABELS[key])
for ax, ylabel, title in [
    (ax_tp, 'Throughput (spec/s)', 'Throughput vs Batch Size'),
    (ax_lat, 'ms / spectrum',      'Latency vs Batch Size'),
]:
    ax.set_xticks(_xi); ax.set_xticklabels(_xlbl)
    ax.set_xlabel('Batch size'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=8, frameon=False); ax.spines[['top','right']].set_visible(False)
ax_lat.axhline(10, color='black', lw=1.5, ls=':', label='10ms (100Hz)')
ax_lat.axhline(35, color='purple', lw=1.2, ls='--', label='35ms (29Hz)')
ax_lat.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig('results/throughput_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: results/throughput_comparison.png')

# ── Figure 3 — Latency distribution histograms (bs=1) ────────────────
fig3, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig3.suptitle('Latency Distribution (bs=1, 5000 spectra) — Three Variants', fontweight='bold')
for ax, (key, timing) in zip(axes, [('fp32',timing_fp32),('bf16',timing_bf16),('compiled',timing_compiled)]):
    _raw = timing[1]['raw']['total']
    ax.hist(_raw, bins=30, color=COLORS[key], alpha=0.8, edgecolor='none')
    ax.axvline(np.mean(_raw), color='black', lw=2, ls='--',
               label=f'mean={np.mean(_raw):.1f}ms')
    ax.axvline(10, color='red', lw=1.5, ls=':', label='10ms target')
    ax.set_title(LABELS[key]); ax.set_xlabel('ms / spectrum')
    ax.legend(fontsize=8, frameon=False); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/latency_histograms.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: results/latency_histograms.png')

Saved: results/stage_comparison.png
Saved: results/throughput_comparison.png
Saved: results/latency_histograms.png


In [10]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — Summary Report
# ═══════════════════════════════════════════════════════════════════════
_now  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
_spd  = lambda a, b: a['total'] / max(b['total'], 0.001)

summary = f"""CASANOVO NAR PROFILING — Three-Way Comparison
Generated  : {_now}
Hardware   : {GPU_NAME} | {TOTAL_VRAM:.1f} GB VRAM | PyTorch {torch.__version__}
depthcharge: {_dc.__file__}  (local branch — nar-flash-cuda-graph-compat)

DEPTHCHARGE CHANGES IN THIS BRANCH
  analytes.py : AnalyteTransformerDecoder.embed() + forward()
    — New param flash_compatible=False (fully backward-compatible)
    — When True: suppresses causal tgt_mask + tgt_key_padding_mask
    — Enables PyTorch FlashAttention SDPA backend for decoder self-attention
  spectra.py  : SpectrumTransformerEncoder.forward()
    — Replaced torch.tensor([[False]]*batch_size) with new_zeros()
    — Eliminates data-dependent Python control flow that caused
      TorchDynamo graph breaks in CUDA Graph capture (was 4 fragments)

NAR PATCH (Cell 0) CHANGE
  Old: injected torch.zeros((L,L), dtype=torch.bool) as tgt_mask
  New: passes flash_compatible=True → parent handles everything cleanly

FLASHATTENTION PROBE RESULT
  flash_compatible=True + BF16 : {'ACCEPTED ✓' if FLASH_ACCEPTED else 'REJECTED — see profiler for actual kernel used'}
  {_attn_a}
  {_attn_b}
  {_attn_c}

KERNEL LAUNCH COMPARISON (bs=1, 50 profiled spectra)
  FP32 baseline          : {_nkern_a} launches/spectrum
  BF16 + Flash           : {_nkern_b} launches/spectrum
  Compiled + BF16 + Flash: {_nkern_c} launches/spectrum

TORCH-COMPILED REGIONS (bs=1 profiler)
  FP32 baseline          : {_nreg_a or 'N/A'} regions (eager — expected N/A)
  BF16 + Flash           : {_nreg_b or 'N/A'} regions (eager — expected N/A)
  Compiled + BF16 + Flash: {_nreg_c} regions  (was 4 before spectra.py fix; expect 1-2 now)

TIMING RESULTS (real spectra, 5000 per batch size)
{"Batch":>6} {"FP32(ms)":>10} {"BF16+Flash(ms)":>16} {"Compiled(ms)":>14} {"BF16 vs FP32":>14} {"Comp vs FP32":>14}
{"-"*76}
"""
for bs in BATCH_SIZES:
    f=timing_fp32[bs]['total']; b=timing_bf16[bs]['total']; c=timing_compiled[bs]['total']
    summary += f'{bs:>6} {f:>10.2f} {b:>16.2f} {c:>14.2f} {f/max(b,0.001):>13.2f}× {f/max(c,0.001):>13.2f}×\n'

summary += f"""
STAGE BREAKDOWN (bs=1, 5000 spectra)
FP32 Baseline:
{df_stage_fp32.to_string(index=False)}

BF16 + Flash:
{df_stage_bf16.to_string(index=False)}

Compiled + BF16 + Flash:
{df_stage_comp.to_string(index=False)}

GPU UTILIZATION
  FP32 baseline          : {gpu_util_fp32:.0f}% util | {gpu_vram_fp32:.2f} GB VRAM
  BF16 + Flash           : {gpu_util_bf16:.0f}% util | {gpu_vram_bf16:.2f} GB VRAM
  Compiled + BF16 + Flash: {gpu_util_comp:.0f}% util | {gpu_vram_comp:.2f} GB VRAM

COMPILE COST (one-time, excluded from timings)
{df_tp_comp[['batch_size','compile_s']].to_string(index=False)}

ARTIFACTS
  results/stage_comparison.png
  results/throughput_comparison.png
  results/latency_histograms.png
  results/trace_fp32.json | results/trace_bf16.json | results/trace_compiled.json
  results/profiler_fp32.txt | results/profiler_bf16.txt | results/profiler_compiled.txt
  results/fp32_stage_bs1.csv | results/bf16_stage_bs1.csv | results/compiled_stage_bs1.csv
  results/fp32_throughput.csv | results/bf16_throughput.csv | results/compiled_throughput.csv
  results/nar_summary.txt
"""
print(summary)
with open('results/nar_summary.txt', 'w') as f: f.write(summary)

print('\n── results/ ──')
for _f in sorted(os.listdir('results')):
    _fp = os.path.join('results', _f)
    print(f'  {_f:<50} {os.path.getsize(_fp)/1024:.1f} KB')
print('\nProfiling complete.')

CASANOVO NAR PROFILING — Three-Way Comparison
Generated  : 2026-06-30 10:53
Hardware   : NVIDIA L4 | 23.6 GB VRAM | PyTorch 2.7.1+cu128
depthcharge: /teamspace/studios/this_studio/depthcharge_changes/depthcharge/__init__.py  (local branch — nar-flash-cuda-graph-compat)

DEPTHCHARGE CHANGES IN THIS BRANCH
  analytes.py : AnalyteTransformerDecoder.embed() + forward()
    — New param flash_compatible=False (fully backward-compatible)
    — When True: suppresses causal tgt_mask + tgt_key_padding_mask
    — Enables PyTorch FlashAttention SDPA backend for decoder self-attention
  spectra.py  : SpectrumTransformerEncoder.forward()
    — Replaced torch.tensor([[False]]*batch_size) with new_zeros()
    — Eliminates data-dependent Python control flow that caused
      TorchDynamo graph breaks in CUDA Graph capture (was 4 fragments)

NAR PATCH (Cell 0) CHANGE
  Old: injected torch.zeros((L,L), dtype=torch.bool) as tgt_mask
  New: passes flash_compatible=True → parent handles everything cleanly

F